In [ ]:
import matplotlib.pyplot as plt

import numpy as np

import shutil
import os

from pathlib import Path

import hdf5storage

from probeinterface import Probe
from probeinterface.plotting import plot_probe

import spikeinterface.full as si

global_job_kwargs = dict(n_jobs=-1, chunk_duration="1s")
si.set_global_job_kwargs(**global_job_kwargs)

## load spikeGLX data and sync .imec and .nidq

In [ ]:
# Load spikeGLX data using SpikeInterface
session_folder_path = '<session_folder_path>'
session_name = '<session_name>'
file_path = os.path.join(session_folder_path, session_name)

# motion folder same path
motion_folder = os.path.join(file_path, 'motion_folder_dredge')

In [ ]:
%matplotlib inline

# Load recordings
# fs = 30000 for imec0.ap, fs = 10000 for nidq
# load nidq
raw_nidq_recording = si.read_spikeglx(file_path,stream_name='nidq')
# need to additionally get the sync channel from imec0.ap, temporally use load_sync_channel=True first
raw_imec_ap_recording = si.read_spikeglx(file_path,stream_name='imec0.ap',load_sync_channel=True)


################################## 
# Usually, imec.ap and nidq are not synced, imec.ap is delayed compared to nidq.
# Note that the sync signal are the same in physical world, but recorded at different time points in the two files, so we need to align them 
##################################

# synchronize_neuropixel_streams(), spikeinterface function comming soon
batch_sync_second = 10  # first ?? seconds to use for sync channel

# Get the sync channel from both recordings, the sync channel is the last channel
imec_sync_channel = raw_imec_ap_recording.get_traces(start_frame=0, end_frame=int(batch_sync_second*raw_imec_ap_recording.get_sampling_frequency()), channel_ids=[raw_imec_ap_recording.get_channel_ids()[-1]]) 
nidq_sync_channel = raw_nidq_recording.get_traces(start_frame=0, end_frame=int(batch_sync_second*raw_nidq_recording.get_sampling_frequency()), channel_ids=[raw_nidq_recording.get_channel_ids()[-1]])

batch_imec_time = raw_imec_ap_recording.get_times()[0:int(batch_sync_second*raw_imec_ap_recording.get_sampling_frequency())]  # in seconds
batch_imec_time = np.array(batch_imec_time)
batch_nidq_time = raw_nidq_recording.get_times()[0:int(batch_sync_second*raw_nidq_recording.get_sampling_frequency())]  # in seconds
batch_nidq_time = np.array(batch_nidq_time)

# normalize the sync channels
imec_sync_channel = (imec_sync_channel - np.min(imec_sync_channel)) / (np.max(imec_sync_channel) - np.min(imec_sync_channel))
nidq_sync_channel = (nidq_sync_channel - np.min(nidq_sync_channel)) / (np.max(nidq_sync_channel) - np.min(nidq_sync_channel))


# Find the rising edge of the sync channel in both recordings
imec_sync_rising_edge = np.where(np.diff(imec_sync_channel[:,0]) > 0.5)[0]
nidq_sync_rising_edge = np.where(np.diff(nidq_sync_channel[:,0]) > 0.5)[0]

# Calculate the time difference between the two sync channels
time_diff = np.mean(batch_nidq_time[nidq_sync_rising_edge] - batch_imec_time[imec_sync_rising_edge])

# Shift the nidq recording
synced_nidq_times_whole = raw_nidq_recording.get_times() - time_diff
raw_nidq_recording.set_times(synced_nidq_times_whole)

# now get imec0.ap use load_sync_channel=False first
raw_imec_ap_recording = si.read_spikeglx(file_path,stream_name='imec0.ap',load_sync_channel=False)

# now plot and show that the sync channels are aligned after synchronization
synced_batch_imec_time = raw_imec_ap_recording.get_times()[0:int(batch_sync_second*raw_imec_ap_recording.get_sampling_frequency())]  # in seconds
synced_batch_imec_time = np.array(synced_batch_imec_time)
synced_batch_nidq_time = raw_nidq_recording.get_times()[0:int(batch_sync_second*raw_nidq_recording.get_sampling_frequency())]  # in seconds
synced_batch_nidq_time = np.array(synced_batch_nidq_time)

plt.figure(figsize=(10,5))
# imec_time and nidq_time are not aligned before sync
plt.subplot(4,1,1)
plt.title('Before sync - time')
plt.plot(batch_imec_time,imec_sync_channel,'r',label='imec.ap sync channel')
plt.plot(batch_nidq_time,nidq_sync_channel,'k',label='nidq sync channel')
plt.xlabel('Time (s)')
plt.legend()

# here shows that if only counting on samples, the two sync channels are aligned
plt.subplot(4,1,2)
plt.title('Before sync - samples')
plt.plot(np.arange((len(imec_sync_channel)))/30000,imec_sync_channel,'r')
plt.plot(np.arange((len(nidq_sync_channel)))/10000,nidq_sync_channel,'k')
plt.xlabel('# samples')

plt.subplot(4,1,3)
plt.title('After sync - time')
plt.plot(synced_batch_imec_time,imec_sync_channel,'r',label='imec.ap sync channel')
# plt.plot(synced_batch_nidq_time,nidq_sync_channel,'k',label='nidq sync channel')
plt.xlabel('Time (s)')
plt.legend()

plt.subplot(4,1,4)
plt.title('After sync - time')
# plt.plot(synced_batch_imec_time,imec_sync_channel,'r',label='imec.ap sync channel')
plt.plot(synced_batch_nidq_time,nidq_sync_channel,'k',label='nidq sync channel')
plt.xlabel('Time (s)')
plt.legend()
plt.show()


In [ ]:
# check imec ap
raw_imec_ap_recording


In [ ]:
# check nidq
raw_nidq_recording


In [ ]:
# add probe and channel info

# NPXL already has channal map
print(raw_imec_ap_recording.get_channel_ids())
print(raw_imec_ap_recording.get_channel_locations())


## Prepare data for ks4 spike sorting

In [ ]:
# preprocess
def preprocess_chain(rec):
	rec = rec.astype('float32')
	rec = si.phase_shift(rec)
	rec = si.common_reference(rec, reference="global", operator="median")
	rec = si.bandpass_filter(rec, freq_min=300.0, freq_max=6000.0)
	return rec

processed_imec_ap_recording = preprocess_chain(raw_imec_ap_recording)

In [ ]:
# DREDge drift correction

job_kwargs = dict(n_jobs=-1, chunk_duration="1s", progress_bar=True)

if Path(motion_folder).exists():
	print("The file exists, please check it first.")
	# shutil.rmtree(motion_folder)
motion, motion_info = si.compute_motion(processed_imec_ap_recording, preset='dredge', folder=motion_folder, output_motion_info=True, **job_kwargs)

In [ ]:
# plot drift motion
motion_info = si.load_motion_info(motion_folder)

# and plot
fig = plt.figure(figsize=(14, 8))
si.plot_motion_info(
	motion_info, processed_imec_ap_recording,
	figure=fig,
	color_amplitude=True,
	amplitude_cmap="inferno",
	scatter_decimate=10,
)

fig.suptitle("dredge")

In [ ]:
# get interpolated_recording
from spikeinterface.sortingcomponents.motion import interpolate_motion

motion_info = si.load_motion_info(motion_folder)
motion = motion_info['motion']
interpolated_recording = interpolate_motion(recording=processed_imec_ap_recording, motion=motion)

# save interpolated recording
interpolated_recording.save(folder=os.path.join(motion_folder,'interpolated_recording'), format='binary', dtype="int16")


In [ ]:
# check interpolated_recording
interpolated_recording = si.load(os.path.join(motion_folder,'interpolated_recording'))

interpolated_recording

In [ ]:
# need to save the probe into _kilosortChanMap.mat for ks4

# get new channel map for the interpolated recording

N_interpolated_channel = interpolated_recording.get_num_channels()

N_channel = len(processed_imec_ap_recording.get_channel_ids())
neuropixel_index = np.arange(N_channel)
interpolated_neuropixel_index = np.zeros((N_interpolated_channel,))
for i in range(N_interpolated_channel):
	interpolated_neuropixel_index[i] = neuropixel_index[processed_imec_ap_recording.get_channel_ids().tolist().index(interpolated_recording.get_channel_ids()[i])]
interpolated_neuropixel_index = interpolated_neuropixel_index - np.min(interpolated_neuropixel_index)
# return index of the sorted array, so that we can use it to sort the channel locations
interpolated_neuropixel_index = np.argsort(interpolated_neuropixel_index) # np.argsort(np.argsort(interpolated_neuropixel_index))
interpolated_neuropixel_index = interpolated_neuropixel_index.astype(int)

print(interpolated_recording.get_channel_ids())
# print(interpolated_recording.get_channel_locations())
print(interpolated_recording.get_channel_locations()[interpolated_neuropixel_index,:])

# interpolated_recording.set_dummy_probe_from_locations(interpolated_recording.get_channel_locations())


from probeinterface.plotting import plot_probe
plot_probe(interpolated_recording.get_probe())


# save to _kilosortChanMap.mat for KS4 to run
from scipy.io import savemat

data_to_save = {
	'chanMap': np.array(np.arange(len(interpolated_neuropixel_index))+1).T,
	'chanMap0ind': np.array(np.arange(len(interpolated_neuropixel_index))).T,
	'connected': np.array([np.ones(N_interpolated_channel)]).astype(bool).T,
	'kcoords': np.array([np.ones(N_interpolated_channel)]).T,
	'name': 'interpolated',
	'xcoords': interpolated_recording.get_channel_locations()[interpolated_neuropixel_index,0].reshape(-1,1),
	'ycoords': interpolated_recording.get_channel_locations()[interpolated_neuropixel_index,1].reshape(-1,1)
}
savemat(Path(motion_folder) / 'interpolated_kilosortChanMap.mat', data_to_save)

## Run kilosort4

In [ ]:
from kilosort import run_kilosort, DEFAULT_SETTINGS


settings = DEFAULT_SETTINGS.copy()
settings['data_dir'] = os.path.join(motion_folder,'interpolated_recording')
settings['n_chan_bin'] = interpolated_recording.get_num_channels()

#  assmume int16 data

print(settings['data_dir'])
ops, st, clu, tF, Wall, similar_templates, is_ref, est_contam_rate, kept_spikes = run_kilosort(settings=settings,
																							   probe_name=Path(motion_folder) / 'interpolated_kilosortChanMap.mat')